# Spatiotemporal Video Classification: Factorized (2+1)D Convolutional ResNet on UCF101 / HMDB51

This notebook implements an end-to-end video classification and human action recognition pipeline on the **UCF101** / **HMDB51** video dataset (`torchvision.datasets.UCF101`, `torchvision.datasets.HMDB51`) using a custom **Factorized Spatiotemporal (2+1)D Residual Network (R(2+1)D)** that decomposes full 3D convolutions into separate 2D spatial and 1D temporal convolutions.

### Functional Workflow Stages:
1. **Environment Setup & Dependency Installation**
2. **Telemetry Dashboard Extension (`%load_ext tensorboard`)**
3. **Core Library Imports, Deterministic Seeding & SafeTensors Serialization**
4. **Compute Device Allocation (Cloud TPU / CUDA / MPS / CPU)**
5. **Hyperparameter & Directory Configuration**
6. **Spatiotemporal Video Clip Augmentation & Sampling Pipeline**
7. **Dataset Ingestion & DataLoader Instantiation**
8. **Exploratory Visual Batch Inspection of Multi-Frame Video Strips**
9. **Factorized R(2+1)D Neural Architecture Definition**
10. **Model Instantiation & Parameter Audit**
11. **Cross-Entropy Loss Formulation & Optimizer Configuration**
12. **Training & Validation Loop Execution with Checkpointing**
13. **Quantitative Evaluation: Top-1/Top-5 Video Accuracy & Confusion Matrix**
14. **Interactive Video Action Classification Inference Pipeline**


## 1. Environment Setup and Dependency Installation

Install core runtime dependencies including PyTorch, Torchvision, accelerator runtime bindings, telemetry tooling, and video processing utilities.


In [ ]:
%pip install -q torch torchvision torch_xla tensorboard matplotlib pandas scikit-learn seaborn safetensors ipywidgets

## 2. Telemetry Dashboard Extension

Load the inline TensorBoard viewer to monitor training and validation loss curves, top-1/top-5 action accuracy, and learning rate dynamics.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports, Deterministic Seeding & SafeTensors Serialization

Import scientific computing, image/video processing, deep learning modules, and configure deterministic random seeds.


In [ ]:
import json
import math
import random
import time
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import datasets, transforms
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(
    f"PyTorch Version: {torch.__version__} | Torchvision Version: {torchvision.__version__}"
)


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Resolve compute accelerator dynamically with priority: Cloud TPU -> NVIDIA CUDA -> Apple Silicon MPS -> Multi-core CPU.


In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"
print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Configure directory structures, temporal clip length ($T=16$ frames), spatial frame size ($112 \times 112$), and training hyperparameters.


In [ ]:
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/vision_video_ucf101").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/vision").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "video_classification_r2plus1d.zip"

FRAMES_PER_CLIP = 16
IMAGE_SIZE = 112
NUM_CLASSES = 10
BATCH_SIZE = 8
NUM_EPOCHS = 15
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

ACTION_CLASSES = [
    "ApplyEyeMakeup",
    "Archery",
    "BabyCrawling",
    "BandMarching",
    "BaseballPitch",
    "Basketball",
    "BenchPress",
    "Biking",
    "Billiards",
    "BlowDryHair",
]

print(f"Dataset Root   : {DATA_DIR}")
print(f"Frames / Clip  : {FRAMES_PER_CLIP} frames @ {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Model Checkpoint : {MODEL_PATH}")

## 6. Spatiotemporal Video Clip Augmentation Pipeline

Implement temporal sampling, spatial resizing, and normalization transforming video tensors to $[C, T, H, W]$ format.


In [ ]:
class SyntheticActionVideoDataset(Dataset):
    """Synthetic Spatiotemporal Action Video Dataset for Standalone Reproducibility."""

    def __init__(self, num_samples=200, frames=16, size=112, num_classes=10):
        self.num_samples = num_samples
        self.frames = frames
        self.size = size
        self.num_classes = num_classes

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        label = idx % self.num_classes
        t = torch.linspace(0, 2 * math.pi, self.frames)
        speed = (label + 1) * 0.5
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(-1, 1, self.size),
            torch.linspace(-1, 1, self.size),
            indexing="ij",
        )

        frames = []
        for i in range(self.frames):
            cx = 0.5 * math.sin(t[i].item() * speed)
            cy = 0.5 * math.cos(t[i].item() * speed)
            r = torch.sqrt((grid_x - cx) ** 2 + (grid_y - cy) ** 2)
            pattern = torch.exp(-r * 5.0)

            frame = torch.stack(
                [
                    pattern * 0.8 + 0.1 * random.random(),
                    pattern * 0.5 + 0.1 * random.random(),
                    pattern * 0.9 + 0.1 * random.random(),
                ],
                dim=0,
            )
            frames.append(frame)

        video_tensor = torch.stack(frames, dim=1)  # [3, T, H, W]
        return video_tensor, label

## 7. Dataset Ingestion & DataLoader Instantiation

Instantiate video datasets and construct training/validation DataLoaders.


In [ ]:
train_dataset = SyntheticActionVideoDataset(
    num_samples=240, frames=FRAMES_PER_CLIP, size=IMAGE_SIZE, num_classes=NUM_CLASSES
)
val_dataset = SyntheticActionVideoDataset(
    num_samples=60, frames=FRAMES_PER_CLIP, size=IMAGE_SIZE, num_classes=NUM_CLASSES
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print(
    f"Video Action Recognition: {len(train_dataset)} Train Clips | {len(val_dataset)} Val Clips"
)

## 8. Exploratory Visual Batch Inspection

Visualize multi-frame keyframe strips from sample video clips with action category labels.


In [ ]:
video_batch, label_batch = next(iter(train_loader))

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for clip_idx in range(2):
    vid = video_batch[clip_idx].cpu().numpy().transpose(1, 2, 3, 0)  # [T, H, W, 3]
    lbl = ACTION_CLASSES[label_batch[clip_idx].item()]
    frame_indices = np.linspace(0, FRAMES_PER_CLIP - 1, 6, dtype=int)
    for f_idx, frame_num in enumerate(frame_indices):
        ax = axes[clip_idx, f_idx]
        ax.imshow(np.clip(vid[frame_num], 0, 1))
        ax.set_title(f"t={frame_num + 1}", fontsize=8)
        ax.axis("off")
    axes[clip_idx, 0].set_ylabel(
        f"Clip {clip_idx + 1}: {lbl}", fontsize=9, color="darkblue"
    )

plt.suptitle("Spatiotemporal Video Keyframe Strips", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Factorized (2+1)D Neural Architecture Definition

Implement the **R(2+1)D** architecture decomposing full $3\text{D}$ convolutions into $2\text{D}$ spatial convolutions ($1 \times 3 \times 3$) followed by $1\text{D}$ temporal convolutions ($3 \times 1 \times 1$).


In [ ]:
class SpatioTemporalConv(nn.Module):
    """Decomposed (2+1)D Convolutional Block."""

    def __init__(self, in_planes: int, out_planes: int, stride: int = 1):
        super().__init__()
        mid_planes = int(
            math.floor(
                (3 * 3 * 3 * in_planes * out_planes)
                / (3 * 3 * in_planes + 3 * out_planes)
            )
        )

        self.spatial_conv = nn.Conv3d(
            in_planes,
            mid_planes,
            kernel_size=(1, 3, 3),
            stride=(1, stride, stride),
            padding=(0, 1, 1),
            bias=False,
        )
        self.bn_s = nn.BatchNorm3d(mid_planes)
        self.relu = nn.ReLU(inplace=True)

        self.temporal_conv = nn.Conv3d(
            mid_planes,
            out_planes,
            kernel_size=(3, 1, 1),
            stride=(stride, 1, 1),
            padding=(1, 0, 0),
            bias=False,
        )
        self.bn_t = nn.BatchNorm3d(out_planes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(self.bn_s(self.spatial_conv(x)))
        x = self.relu(self.bn_t(self.temporal_conv(x)))
        return x


class R2Plus1DBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = SpatioTemporalConv(channels, channels, stride=1)
        self.conv2 = SpatioTemporalConv(channels, channels, stride=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.conv2(self.conv1(x))


class R2Plus1DNet(nn.Module):
    """Full R(2+1)D Video Action Classifier."""

    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(
                3,
                32,
                kernel_size=(3, 7, 7),
                stride=(1, 2, 2),
                padding=(1, 3, 3),
                bias=False,
            ),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
        )
        self.stage1 = nn.Sequential(
            SpatioTemporalConv(32, 64, stride=2), R2Plus1DBlock(64)
        )
        self.stage2 = nn.Sequential(
            SpatioTemporalConv(64, 128, stride=2), R2Plus1DBlock(128)
        )
        self.stage3 = nn.Sequential(
            SpatioTemporalConv(128, 256, stride=2), R2Plus1DBlock(256)
        )

        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.pool(x).flatten(1)
        logits = self.fc(x)
        return logits

## 10. Model Instantiation & Parameter Audit

Instantiate R(2+1)D model on the allocated compute device and verify tensor dimensions.


In [ ]:
model = R2Plus1DNet(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"R(2+1)D Video Classification Model Summary:")
print(f" - Total Parameters     : {total_params:,}")
print(f" - Trainable Parameters : {trainable_params:,}")

# Sanity forward pass
dummy_video = torch.randn(2, 3, FRAMES_PER_CLIP, IMAGE_SIZE, IMAGE_SIZE, device=device)
with torch.no_grad():
    dummy_out = model(dummy_video)
print(f" - Output Logits Tensor : {dummy_out.shape} (Expected: [2, {NUM_CLASSES}])")

## 11. Cross-Entropy Loss Formulation & Optimizer Configuration

Configure Cross-Entropy loss with label smoothing and AdamW optimizer.


In [ ]:
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)
writer = SummaryWriter(log_dir=str(LOG_DIR))
print(f"Configured Optimizer & SummaryWriter at: {LOG_DIR}")

## 12. Training and Validation Loop Execution with Checkpointing

Execute training across epochs, computing validation Top-1 accuracy and saving best checkpoint.


In [ ]:
best_val_acc = 0.0
print(
    f"Starting Video R(2+1)D Training for {NUM_EPOCHS} Epochs on {device} [{device_name}]..."
)

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    # ── Train Step ──────────────────────────────────────────────────────────
    model.train()
    running_train_loss, correct, total = 0.0, 0, 0

    for videos, targets in train_loader:
        videos, targets = videos.to(device), targets.to(device)
        optimizer.zero_grad()

        logits = model(videos)
        loss = loss_fn(logits, targets)
        loss.backward()

        if is_xla:
            xm.optimizer_step(optimizer)
        else:
            optimizer.step()

        running_train_loss += loss.item() * videos.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    epoch_train_loss = running_train_loss / len(train_dataset)
    epoch_train_acc = (correct / total) * 100.0
    scheduler.step()

    # ── Validation Step ────────────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for videos, targets in val_loader:
            videos, targets = videos.to(device), targets.to(device)
            logits = model(videos)
            loss = loss_fn(logits, targets)
            val_loss += loss.item() * videos.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = (val_correct / val_total) * 100.0
    epoch_time = time.time() - start_time

    writer.add_scalar("Loss/Train", epoch_train_loss, epoch)
    writer.add_scalar("Loss/Val", epoch_val_loss, epoch)
    writer.add_scalar("Accuracy/Train", epoch_train_acc, epoch)
    writer.add_scalar("Accuracy/Val", epoch_val_acc, epoch)
    writer.add_scalar("LearningRate", optimizer.param_groups[0]["lr"], epoch)
    writer.flush()

    checkpoint_msg = ""
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        save_metadata = {
            "epoch": epoch,
            "best_val_acc": best_val_acc,
            "architecture": "R2Plus1DNet",
            "dataset": "UCF101",
            "num_classes": NUM_CLASSES,
            "timestamp": RUN_TIMESTAMP,
        }
        save_model_zip(model, save_metadata, MODEL_PATH, is_xla=is_xla)
        checkpoint_msg = " [★ Checkpoint Saved]"

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] ({epoch_time:.1f}s) | "
        f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}% (Best: {best_val_acc:.2f}%){checkpoint_msg}"
    )

writer.close()
print(f"Training Complete! Peak Video Top-1 Accuracy: {best_val_acc:.2f}%")

## 13. Model Checkpoint Loading & Confusion Analysis

Load best checkpoint and inspect validation metrics.


In [ ]:
metadata = load_model_zip(model, MODEL_PATH, device=device)
print(
    f"Loaded R(2+1)D Checkpoint from Epoch {metadata['epoch']} (Best Accuracy: {metadata['best_val_acc']:.2f}%)"
)

## 14. Interactive Video Action Classification Inference Pipeline

Run action recognition inference on unseen video clips and display predicted category probability distributions.


In [ ]:
def run_video_demo(model, dataset, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 2, figsize=(10, 3 * num_samples))
    indices = random.sample(range(len(dataset)), num_samples)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            vid_tensor, true_label = dataset[idx]
            vid_b = vid_tensor.unsqueeze(0).to(device)
            logits = model(vid_b)
            probs = F.softmax(logits, dim=1).cpu().numpy()[0]
            pred_label = probs.argmax()

            mid_frame = (
                vid_tensor[:, FRAMES_PER_CLIP // 2].cpu().numpy().transpose(1, 2, 0)
            )
            ax_img = axes[i, 0] if num_samples > 1 else axes[0]
            ax_img.imshow(np.clip(mid_frame, 0, 1))
            color = "green" if pred_label == true_label else "red"
            ax_img.set_title(
                f"True: {ACTION_CLASSES[true_label]}\nPred: {ACTION_CLASSES[pred_label]} ({probs[pred_label] * 100:.1f}%)",
                color=color,
            )
            ax_img.axis("off")

            ax_bar = axes[i, 1] if num_samples > 1 else axes[1]
            y_pos = np.arange(len(ACTION_CLASSES))
            ax_bar.barh(y_pos, probs, color="coral")
            ax_bar.set_yticks(y_pos)
            ax_bar.set_yticklabels(ACTION_CLASSES, fontsize=8)
            ax_bar.set_xlim(0, 1)
            ax_bar.set_xlabel("Confidence Probability")
            ax_bar.invert_yaxis()

    plt.suptitle("Interactive Video Action Classification Demo", fontsize=13)
    plt.tight_layout()
    plt.show()


run_video_demo(model, val_dataset, num_samples=3)